In [380]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [381]:
batch_size = 128
test_size = 1000
learning_rate = 0.001
epochs = 10

input_dim = 28
conv1_channels = 16
conv2_channels = 32
kernel_size = 3
stride = 2
pool_size = 2

In [382]:
dataset_train = datasets.MNIST('data', train=True, download=True, transform=transforms.ToTensor())
dataset_test = datasets.MNIST('data', train=False, transform=transforms.ToTensor())

train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset_test, batch_size=test_size, shuffle=False)

In [383]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, 1)  # 卷积层1: 1->16通道
        self.conv2 = nn.Conv2d(16, 32, 3, 1)  # 卷积层2: 16->32通道
        self.pool = nn.MaxPool2d(2, 2)  # 2x2 池化
        self.fc1 = nn.Linear(32 * 5 * 5, 128)  # 计算池化后的大小 (32,5,5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [384]:
model = CNN()

In [385]:
# criterion = nn.CrossEntropyLoss()
criterion = nn.NLLLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [386]:
def train(model, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        output = model(data)
        loss = criterion(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

In [387]:
def test(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            loss = criterion(output, target)
            test_loss += loss.item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({100. * correct / len(test_loader.dataset):.0f}%)\n')

In [388]:
for epoch in range(1, epochs + 1):
    train(model, train_loader, optimizer, epoch)
    test(model, test_loader)
    optimizer.step()

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.308909
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.273173
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.154360
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.135233
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.056401

Test set: Average loss: 0.0001, Accuracy: 9728/10000 (97%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 0.079930
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.052158
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.031598
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.072519
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.049540

Test set: Average loss: 0.0001, Accuracy: 9837/10000 (98%)

Train Epoch: 3 [0/60000 (0%)]	Loss: 0.056357
Train Epoch: 3 [12800/60000 (21%)]	Loss: 0.016798
Train Epoch: 3 [25600/60000 (43%)]	Loss: 0.075205
Train Epoch: 3 [38400/60000 (64%)]	Loss: 0.114840
Train Epoch: 3 [51200/60000 (85%)]	Loss: 0.028307

Test set: Average loss: 0.0000, Accuracy: 9861/10000 (99%)

Train Epoch: 4 [0/60000 (0%)]	Loss: 0.068860
Train Epoch: 4 [12800/60000 (21%)]	Lo